# setup + imports

In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


/Users/ecaterina/practice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 17:04:31.918000 73382 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [4]:
fit = train_data.sample(frac=0.9, random_state=0)
val = train_data.drop(fit.index)

X = list(fit['Text'])
y = list(fit['RevisedText'])
X_val = list(val['Text'])
y_val = list(val['RevisedText'])

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 4728.23it/s]


# validare

In [6]:
def cosim(v1, v2):
    return v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2))

def levenshtein(a, b):
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i
    for j in range(len(b) + 1):
        d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,         # delete
                          d[i][j - 1] + 1,         # insert
                          d[i - 1][j - 1] + cost)  # replace or keep
    return d[len(a)][len(b)]


def finalscore(pred, ref):
    ep = model.encode(pred)
    et = model.encode(ref)
    edit = 1 - levenshtein(pred, ref) / max(len(pred), len(ref), 1)
    return 0.7*cosim(ep, et) + 0.3*edit

def evaluate(preds, refs):
    scores = []
    for i in range(len(preds)):
        scores.append(finalscore(preds[i], refs[i]))
    return np.mean(scores)

In [7]:
print(evaluate(list(X_val), list(y_val)))

0.9338157


# model

In [8]:
tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5  = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(device)

Loading weights: 100%|█████████████████████| 190/190 [00:00<00:00, 4356.91it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [9]:
def make_batch(texts, targets=None):
    tt = []
    for t in texts:
        tt.append("grammar: " + t)

    enc = tok(tt, padding=True, truncation=True, max_length=128, return_tensors='pt')
    batch = {
        "input_ids": enc.input_ids.to(device),
        "attention_mask": enc.attention_mask.to(device)
    }
    if targets is not None:
        tt = []
        for t in targets:
            tt.append(t)
        labs = tok(targets, padding=True, truncation=True, max_length=128, return_tensors='pt').input_ids
        labs[labs == tok.pad_token_id] = -100
        batch = {
            "input_ids": enc.input_ids.to(device),
            "attention_mask": enc.attention_mask.to(device),
            "labels": labs.to(device)
        }
    return batch

def gen(texts, batch_size=32):
    t5.eval()
    out = []
    for s in range(0, len(texts), batch_size):
        tt = texts[s : s+batch_size]
        b = make_batch(tt)
        with torch.no_grad():
            gentext = t5.generate(**b, max_new_tokens=128)
        out += tok.batch_decode(gentext, skip_special_tokens=True)
    return out
        

In [10]:
optimizer = torch.optim.AdamW(t5.parameters(), lr=3e-4)

batch_size = 16
best_score = 0
best_state = None

for epoch in range(4):
    t5.train()
    perm = np.random.permutation(len(X))
    total = 0
    for s in range(0, len(X), batch_size):
        idx = perm[s : s + batch_size]
        b = make_batch([X[i] for i in idx], [y[i] for i in idx])
        loss = t5(**b).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    s = evaluate(gen(X_val), y_val)
    print("epoch", epoch, "loss", round(total, 2), "val score", round(s, 4))
    if s > best_score:
        best_score = s
        torch.save(t5.state_dict(), "best.pt") 

t5.load_state_dict(torch.load("best.pt"))

epoch 0 loss 40.76 val score 0.9536
epoch 1 loss 23.88 val score 0.9597
epoch 2 loss 16.73 val score 0.9581
epoch 3 loss 12.37 val score 0.9612


<All keys matched successfully>

In [14]:
def safe(texts, preds, thr):
    out = []
    for t, p in zip(texts, preds):
        change = levenshtein(t, p) / max(len(t), 1)
        if change <= thr:
            out.append(p)
        else:
            out.append(t)
    return out

p_val = gen(X_val)
bestcut = 0
bestcutscore = 0
for m in [0.0, 0.5, 0.75, 0.8, 1.0]:
    s = evaluate(safe(X_val, p_val, m), y_val)
    print(m, round(s, 4))
    if s > bestcutscore:
        bestcutscore = s
        bestcut = m
print(bestcut)

0.0 0.9338
0.5 0.9616
0.75 0.9612
0.8 0.9612
1.0 0.9612
0.5


# submisie

In [17]:
thr = 0.8

X_test = list(test_data['Text'])
preds = safe(X_test, gen(X_test), thr)

answer = pd.DataFrame({
    "DatapointID": test_data["SampleID"],
    "RevisedText": preds
})
     
answer.to_csv('submission.csv', index=False)

In [18]:
preds[:10]

["He didn't eat any breakfast this morning.",
 '29. When I was young, I used to play soccer every day.',
 'The car is parked in front of the house.',
 'Their house is beautiful and far from home.',
 "I can't believe she ate the whole cake herself!",
 'The artist uses paint, charcoal, and collages.',
 'The movie was worse than I expected.',
 'I was waiting for the bus for over an hour.',
 'His watch is expensive.',
 '33. He was driving too fast and lost control of the vehicle.']

In [19]:
print(repr(train_data['Text'].iloc[0]))
print(repr(preds[0]))

'She forgot her umbrella it started to rain.'
"He didn't eat any breakfast this morning."


In [20]:
p_val = gen(X_val)
scores = [finalscore(p_val[i], y_val[i]) for i in range(len(X_val))]
for i in np.argsort(scores)[:10]:
    print(round(scores[i], 3), "|", X_val[i], "|", p_val[i], "|", y_val[i])

0.55 | The smrtph has a 5.5" OLED display. | The smrtph has a 5.5" OLED display. | The smartphone has a 5.5-inch organic light-emitting diode display.
0.604 | The car's fuel eff is rated at 35mpg on hwy. | The car's fuel eff is rated at 35mpg on average. | The car's fuel efficiency is rated at 35 miles per gallon on the highway.
0.686 | She gave a complimentary to her colleague. | She gave her colleague complimentary food. | She gave a compliment to her colleague.
0.737 | She's a prof at UCLA, in the CS dept. | She's a professor at UCLA, in the CS department. | She's a professor at the University of California, Los Angeles, in the Computer Science department.
0.741 | The car was driven by the mechanic for a test with a strange noise. | The mechanic drove the car for a test with a strange noise. | The car with a strange noise was driven by the mechanic for a test.
0.754 | The novel was written by the author set in medieval times. | The novel was written by the author in medieval times. 